# ID10M — System G Zero-Shot Eval\n\nRuns System G (mBERT BIO tagger, trained on MultiIdiom) zero-shot on ID10M test set.  \nNo retraining — uses existing model from Drive.  \nOutputs span F1 + CLS macro F1 derived from BIO predictions.\n\n**Pipeline:**\n1. Mount Drive + clone repo\n2. Clone ID10M dataset\n3. Convert ID10M test TSV → JSONL\n4. Run System G inference on ID10M test\n5. Show results

In [ ]:
# ── CELL 1: Mount Drive + clone repo ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys, os, shutil

REPO     = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
REPO_DIR = '/content/Research_And_Training'

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO, REPO_DIR], check=True)

os.makedirs('/content/drive/MyDrive/Idiomator_Research/models',  exist_ok=True)
os.makedirs('/content/drive/MyDrive/Idiomator_Research/results', exist_ok=True)

for d in ['models', 'results']:
    p = os.path.join(REPO_DIR, d)
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(f'/content/drive/MyDrive/Idiomator_Research/{d}', p)

# HARD GATE
ok = True
for d in ['models', 'results']:
    p = os.path.join(REPO_DIR, d)
    is_link = os.path.islink(p)
    ok &= is_link
    print(d, 'islink:', is_link, '->', os.readlink(p) if is_link else '(REAL DIR - BAD)')
assert ok, 'Symlinks NOT set up — STOP, do not run training cells.'
print('\nGate passed. Safe to proceed.')

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi

In [ ]:
# ── CELL 2: Clone ID10M dataset ───────────────────────────────────────────────
!git clone https://github.com/Babelscape/ID10M /tmp/id10m

# Verify splits exist
from pathlib import Path
for lang in ['english', 'spanish']:
    for split in ['train', 'test']:
        p = Path(f'/tmp/id10m/resources/bio_format/{lang}/{split}_{lang}.tsv')
        print(f'  {p.name}: {"✓" if p.exists() else "MISSING"}')

In [ ]:
# ── CELL 3: Convert ID10M TSV → JSONL ────────────────────────────────────────
# Output goes to Drive for persistence
JSONL_DIR = Path('/content/drive/MyDrive/IdiomatorRigor/id10m_jsonl')
JSONL_DIR.mkdir(parents=True, exist_ok=True)

!python3 -u Additional_Rigor_Experiments/id10m_to_jsonl.py \
    --data_dir  /tmp/id10m \
    --output_dir {JSONL_DIR} \
    --langs EN ES \
    --splits train dev test

# Verify
import json
for split in ['train', 'dev', 'test']:
    p = JSONL_DIR / f'{split}.jsonl'
    if p.exists():
        n = sum(1 for _ in p.open())
        sample = json.loads(p.open().readline())
        print(f'  {split}.jsonl: {n} rows | sample span={sample.get("span_start")}-{sample.get("span_end")} idiom={sample.get("idiom")!r}')

In [ ]:
# ── CELL 5: Run System G zero-shot on ID10M test ─────────────────────────────
import os, subprocess, sys
from pathlib import Path

JSONL_DIR  = Path('/content/drive/MyDrive/IdiomatorRigor/id10m_jsonl')
OUTPUT_DIR = Path('/content/drive/MyDrive/IdiomatorRigor/id10m_eval')
MODEL_DIR  = 'models/bio_tagger_en_es_hi_te'   # trained in cell 4

assert Path(f'/content/Research_And_Training/{MODEL_DIR}/best_model').exists(), \
    f'Model not found — run cell 4 first'
print(f'✓ Model found: {MODEL_DIR}/best_model')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, '-u',
    'Additional_Rigor_Experiments/run_12_id10m_systemg_eval.py',
    '--model_dir',  MODEL_DIR,
    '--data_dir',   str(JSONL_DIR),
    '--output_dir', str(OUTPUT_DIR),
    '--langs',      'English', 'Spanish',
]

proc = subprocess.Popen(
    cmd, cwd='/content/Research_And_Training',
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
assert proc.returncode == 0, 'Eval failed — check log above'

In [ ]:
# ── CELL 4: Train System G on MultiIdiom (~1-2h on T4) ───────────────────────
import torch, os, subprocess, sys
from pathlib import Path

assert torch.cuda.is_available(), 'No CUDA — Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')

# Output goes to models/bio_tagger_en_es_hi_te — symlinked to Drive
MODEL_DIR = 'models/bio_tagger_en_es_hi_te'
MODEL_PATH = Path(f'/content/Research_And_Training/{MODEL_DIR}')
MODEL_PATH.mkdir(parents=True, exist_ok=True)

# Persistence gate
probe = MODEL_PATH / '.write_test'
probe.write_text('ok')
assert probe.read_text() == 'ok', 'OUTPUT NOT DURABLE — check Drive symlink in cell 1'
probe.unlink()
print(f'✓ Output durable: {MODEL_PATH}')

LOG = MODEL_PATH / 'train.log'
cmd = [
    sys.executable, '-u', 'Ablations/BiO_Task_mBERT_train.py',
    '--data_dir',   'idioms_structured/Splits',   # MultiIdiom data in repo
    '--output_dir', MODEL_DIR,
    '--langs',      'English', 'Spanish', 'Hindi', 'Telugu',
    '--test_langs', 'English', 'Spanish', 'Hindi', 'Telugu',
    '--model_name', 'bert-base-multilingual-cased',
    '--epochs',     '6',
    '--batch_size', '32',
    '--lr',         '3.27e-5',
    '--dropout',    '0.2394',
    '--o_weight',   '0.104',
]

with LOG.open('w') as lf:
    proc = subprocess.Popen(
        cmd, cwd='/content/Research_And_Training',
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'}
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
        lf.write(line); lf.flush()
    proc.wait()

assert proc.returncode == 0, 'Training failed'
assert (MODEL_PATH / 'best_model').exists(), 'best_model not saved — check log'
print(f'\n✓ Model saved → {MODEL_PATH}/best_model')

In [ ]:
# ── CELL 5: Results ───────────────────────────────────────────────────────────
import json
from pathlib import Path

results_path = Path('/content/drive/MyDrive/IdiomatorRigor/id10m_eval/id10m_systemg_results.json')
assert results_path.exists(), 'Results not found — cell 4 may have failed'
m = json.loads(results_path.read_text())
print(json.dumps(m, indent=2))

print('\n=== Summary ===')
print(f'{"Lang":<12} {"CLS macro F1":>14} {"Span overlap F1":>16} {"Span exact":>12} {"N":>6}')
print('-' * 64)
for lang, v in m.items():
    print(f'{lang:<12} {v["cls_macro_f1"]:>14.4f} {v["span_overlap_f1"]:>16.4f} {v["span_exact"]:>12.4f} {v["n"]:>6}')

print('\nSystem E zero-shot CLS macro F1 (for comparison):')
print('  EN: 0.6710   ES: 0.5787')